How to design and implement an LLM-powered Chatbot. This chatbot will be able to have a conversation and remember previous interactions. 

In [2]:
import os
from dotenv import load_dotenv
load_dotenv()

groq_api_key = os.getenv('GROQ_API_KEY')

In [5]:
from langchain_groq import ChatGroq

model = ChatGroq(model = 'llama-3.1-8b-instant', groq_api_key = groq_api_key)
model

ChatGroq(profile={'max_input_tokens': 131072, 'max_output_tokens': 8192, 'image_inputs': False, 'audio_inputs': False, 'video_inputs': False, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': False, 'tool_calling': True}, client=<groq.resources.chat.completions.Completions object at 0x15baa47d0>, async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x15baa51d0>, model_name='llama-3.1-8b-instant', model_kwargs={}, groq_api_key=SecretStr('**********'))

In [7]:
from langchain_core.messages import HumanMessage

model.invoke([HumanMessage(content= 'Hi, My name is Axel Heck and I am a Chief AI Engineer.')])


AIMessage(content="Nice to meet you, Axel Heck. It's great to have a Chief AI Engineer like yourself interacting with me. What brings you here today? Are you looking to discuss a specific AI-related topic, or perhaps you'd like to know more about my capabilities? I'm all ears (or rather, all text).", additional_kwargs={}, response_metadata={'token_usage': {'completion_tokens': 64, 'prompt_tokens': 50, 'total_tokens': 114, 'completion_time': 0.09312315, 'completion_tokens_details': None, 'prompt_time': 0.011927346, 'prompt_tokens_details': None, 'queue_time': 0.059039713, 'total_time': 0.105050496}, 'model_name': 'llama-3.1-8b-instant', 'system_fingerprint': 'fp_8f8420ecd7', 'service_tier': 'on_demand', 'finish_reason': 'stop', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--019c343c-4a56-7a73-850f-ddfa80785b29-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 50, 'output_tokens': 64, 'total_tokens': 114})

In [8]:
from langchain_core.messages import AIMessage

model.invoke(
    [
        HumanMessage(content= 'Hi, My name is Axel Heck and I am a Chief AI Engineer.'),
        AIMessage(content= "Nice to meet you, Axel Heck. It's great to have a Chief AI Engineer like yourself interacting with me. What brings you here today? Are you looking to discuss a specific AI-related topic, or perhaps you'd like to know more about my capabilities? I'm all ears (or rather, all text)."),
        HumanMessage(content= 'Hey, What is my name and what do I do?')
    ]
)

AIMessage(content='Your name is Axel Heck, and according to our conversation, you are a Chief AI Engineer.', additional_kwargs={}, response_metadata={'token_usage': {'completion_tokens': 20, 'prompt_tokens': 135, 'total_tokens': 155, 'completion_time': 0.026007535, 'completion_tokens_details': None, 'prompt_time': 0.010379272, 'prompt_tokens_details': None, 'queue_time': 0.058636067, 'total_time': 0.036386807}, 'model_name': 'llama-3.1-8b-instant', 'system_fingerprint': 'fp_020e283281', 'service_tier': 'on_demand', 'finish_reason': 'stop', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--019c3440-3664-7081-a661-695a5253f2b6-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 135, 'output_tokens': 20, 'total_tokens': 155})

### Message History
We can use a Message History class to wrap our model and make it stateful. This will keep track of inputs and outputs of the model, and store them in some datastore. Future interactions will then load those messages and pass them into the chain as part of the input. Let's see how to use this!

In [11]:
from langchain_community.chat_message_histories import ChatMessageHistory
from langchain_core.chat_history import BaseChatMessageHistory
from langchain_core.runnables.history import RunnableWithMessageHistory

store = {}

def get_session_history(session_id: str) -> BaseChatMessageHistory:
    if session_id not in store:
        store[session_id] = ChatMessageHistory()
    return store[session_id]

with_message_history = RunnableWithMessageHistory(model, get_session_history)

In [10]:
config = {'configurable': {"session_id": 'Chat1'}}

In [13]:
response = with_message_history.invoke(
    [
        HumanMessage('Hi, My name is Axel Heck and I am a Chief AI Engineer.')],
        config = config
)

In [14]:
response.content

"It seems like you're repeating your introduction. I've already acknowledged you, Axel. If you're ready to discuss something specific or ask a question, I'm here to help."

In [15]:
with_message_history.invoke(
    [
        HumanMessage('What is my name?')],
        config = config
)

AIMessage(content="Your name is Axel Heck, and you're a Chief AI Engineer.", additional_kwargs={}, response_metadata={'token_usage': {'completion_tokens': 15, 'prompt_tokens': 189, 'total_tokens': 204, 'completion_time': 0.021134934, 'completion_tokens_details': None, 'prompt_time': 0.04956124, 'prompt_tokens_details': None, 'queue_time': 0.061406558, 'total_time': 0.070696174}, 'model_name': 'llama-3.1-8b-instant', 'system_fingerprint': 'fp_d317489708', 'service_tier': 'on_demand', 'finish_reason': 'stop', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--019c3454-fdf0-7171-80f9-59e68b7ecb09-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 189, 'output_tokens': 15, 'total_tokens': 204})

In [17]:
# Change the config -> session_id
config1 = {'configurable': {"session_id": 'Chat2'}}
response = with_message_history.invoke(
    [HumanMessage(content= 'WHat is my name')],
    config = config1
)

response.content

"I don't have any information about your name as our conversation has just started. If you'd like to share your name with me, I'd be happy to chat with you and address you by it."

In [18]:
response = with_message_history.invoke(
    [HumanMessage(content= 'Hey, my name is John')],
    config = config1
)

response.content

'Nice to meet you, John. How are you doing today?'

In [20]:
response = with_message_history.invoke(
    [HumanMessage(content= 'What is my name')],
    config = config1
) 
response.content

"Your name is John. I've already established that. Would you like to talk about something else or is there something specific on your mind?"

In [ ]:
response = with_message_history.batch(
    [
    [HumanMessage(content= 'What is my name')],
    [HumanMessage(content='What skills do I need to have as a professional AI Engineer.')]
    ],
    config= config1
) 
response

[AIMessage(content='John.', additional_kwargs={}, response_metadata={'token_usage': {'completion_tokens': 3, 'prompt_tokens': 315, 'total_tokens': 318, 'completion_time': 0.00905716, 'completion_tokens_details': None, 'prompt_time': 0.079722712, 'prompt_tokens_details': None, 'queue_time': 0.043792287, 'total_time': 0.088779872}, 'model_name': 'llama-3.1-8b-instant', 'system_fingerprint': 'fp_6b5c123dd9', 'service_tier': 'on_demand', 'finish_reason': 'stop', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--019c34a9-10a9-7482-bded-8bfea8c0caac-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 315, 'output_tokens': 3, 'total_tokens': 318}),
 AIMessage(content="As a professional AI Engineer, you'll need to possess a combination of technical skills, business acumen, and soft skills. Here are some key skills to focus on:\n\n**Technical Skills:**\n\n1. **Programming languages:** Python, R, Java, C++, and JavaScript are commonly used in AI development.\n2. **Ma

#### Prompt Template

In [32]:
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder

prompt = ChatPromptTemplate.from_messages(
    [
        ('system', 'You are helpful Assistant. Answer all the questions to the best of your ability.'),
        MessagesPlaceholder(variable_name='messages')
    ]
)

chain = prompt | model

In [33]:
chain.invoke({'messages':[HumanMessage(content='Hi, My name is Axel.')]})

AIMessage(content="Hi Axel, it's nice to meet you. I'm here to help answer any questions or provide information you might need. How can I assist you today?", additional_kwargs={}, response_metadata={'token_usage': {'completion_tokens': 33, 'prompt_tokens': 58, 'total_tokens': 91, 'completion_time': 0.039275734, 'completion_tokens_details': None, 'prompt_time': 0.004793086, 'prompt_tokens_details': None, 'queue_time': 0.060590314, 'total_time': 0.04406882}, 'model_name': 'llama-3.1-8b-instant', 'system_fingerprint': 'fp_020e283281', 'service_tier': 'on_demand', 'finish_reason': 'stop', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--019c34c7-d856-79c3-8c92-b5bb8a27c1eb-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 58, 'output_tokens': 33, 'total_tokens': 91})

In [34]:
with_message_history = RunnableWithMessageHistory(chain, get_session_history)

In [ ]:
# one input variable
config3 = {'configurable': {"session_id": 'Chat3'}}

response = with_message_history.invoke(
    [HumanMessage(content='Hi! my name is Axel Heck.')],
    config= config3,
)

response.content

'It looks like you introduced yourself already, Axel Heck. How are you doing today? Is there something I can help you with or would you like to chat?'

In [37]:
## Adding Multiple input variables

prompt1 = ChatPromptTemplate.from_messages(
    [
        ('system', 'You are helpful Assistant. Answer all the questions to the best of your ability in {language}.'),
        MessagesPlaceholder(variable_name='messages')
    ]
)

chain1 = prompt1 | model

In [38]:
response1 = chain1.invoke({'messages':[HumanMessage(content= ' Hi, my name is Tony Stark.')],
                           'language':'Hindi'})

response1.content

'नमस्ते टोनी स्टार्क जी, मैं आपकी सहायता करने के लिए तैयार हूँ।'

In [44]:
with_message_history4 = RunnableWithMessageHistory(
    chain1,
    get_session_history,
    input_messages_key='messages'
)

In [45]:
config4 = {'configurable': {"session_id": 'Chat4'}}

response4 = with_message_history4.invoke(
    {'messages': [HumanMessage(content="Hi, I am Thor.")], 'language':'Hindi'},
    config= config4
)

response4.content

'क्रिस्त!   Thor, All-Father! आपका स्वागत है! (Krishat! Thor, All-Father! Your welcome is here!)\n\nआपकी सहायता के लिए तैयार हूं। क्या समस्या है? या शायद आप अपने मित्र लोकी और हिडलक के साथ एक नई साहसिक कोशिश करना चाहते हैं?'

#### Managing the Conversation History

'trim_messages' - helper that reduces how many messages are sent to the model. The trimmer allows us to specify how many tokens we want to keep, along with other parameters like if we want to keep the system message and whether to allow partial messages

In [56]:
from langchain_core.messages import SystemMessage, trim_messages

trimmer = trim_messages(
    max_tokens= 45, 
    strategy= 'last',
    token_counter= model,
    include_system= True,
    allow_partial= False,
    start_on= 'human'
)

messages = [
    SystemMessage(content="you're a good assistant"),
    HumanMessage(content="Hi! I'm Bob"),
    AIMessage(content="Hi!"),
    HumanMessage(content= "I like vanilla ice cream."),
    AIMessage(content= "nice "),
    HumanMessage(content= "whats 2 + 2"),
    AIMessage(content= "4 "),
    HumanMessage(content= "Thanks"),
    AIMessage(content= "no problem! "),
    HumanMessage(content= "having fun? "),
    AIMessage(content= "yes! ")
]

trimmer.invoke(messages)

[SystemMessage(content="you're a good assistant", additional_kwargs={}, response_metadata={}),
 HumanMessage(content='whats 2 + 2', additional_kwargs={}, response_metadata={}),
 AIMessage(content='4 ', additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[]),
 HumanMessage(content='Thanks', additional_kwargs={}, response_metadata={}),
 AIMessage(content='no problem! ', additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[]),
 HumanMessage(content='having fun? ', additional_kwargs={}, response_metadata={}),
 AIMessage(content='yes! ', additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[])]

In [57]:
# How to pass trimmer in chain
from operator import itemgetter
from langchain_core.runnables import RunnablePassthrough

chain = (
    RunnablePassthrough.assign(messages = itemgetter('messages') | trimmer )
    | prompt
    | model
)

response = chain.invoke(
    {
    'messages':messages + [HumanMessage(content= 'What I cream do I like?')],
    'language':'English'
    }
)

response.content

'I think you might be asking "what ice cream do I like?" But I\'m not sure yet. Can you tell me what flavors you enjoy?'

we do not get the answer as Vanilla, cause the context length is of 45 so this information might have been trimmed

In [64]:
response = chain.invoke(
    {
    'messages':messages + [HumanMessage(content= 'What math did I ask?')],
    'language':'English'
    }
)

response.content

'You asked a simple math question: 2 + 2.'

In [65]:
response = chain.invoke(
    {
    'messages':messages + [HumanMessage(content= 'What math problem did I ask?')],
    'language':'English'
    }
)

response.content

"You didn't ask a math problem. Our conversation just started with a simple greeting. If you have any questions or need help with a math problem, feel free to ask!"

In [66]:
## Wrapping this in message history

with_message_history5 = RunnableWithMessageHistory(
    chain, 
    get_session_history,
    input_messages_key='messages'
)

config5 = {'configurable': {"session_id": 'Chat5'}}

response = with_message_history5.invoke(
    {
    'messages':messages + [HumanMessage(content= 'What is my name?')],
    'language':'English'
    },
    config=config5
)

response.content


"I don't know your name. You haven't told me yet."

In [67]:
response = with_message_history5.invoke(
    {
    'messages':messages + [HumanMessage(content= 'What math problem did i ask?')],
    'language':'English'
    },
    config=config5
)

response.content

"You didn't ask a math problem. Our conversation started with a simple statement. If you have a math problem you'd like to discuss, I'd be happy to help."